## 01. Notebook purpose

This notebook prepares the dataset for binary classification of machine failure. To avoid leakage, failure-mode columns (TWF, HDF, PWF, OSF, RNF) are excluded from the feature set, as well as identifier columns (UDI, Product ID). The notebook defines preprocessing and limited feature engineering in a reproducible way so the same logic can later be reused in the modeling and MLOps stages.

## 02. Loading Data Set

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn import set_config
set_config(transform_output="pandas")

In [2]:
DATA_PATH = Path("../data/raw/ai4i2020.csv")
df = pd.read_csv(DATA_PATH)

In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
df.shape

(10000, 14)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

## 03. Target Definition and columns exclusion

In [6]:
target_col = "Machine failure"

failure_mode_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
id_cols = ["UDI", "Product ID"]

drop_cols = failure_mode_cols + id_cols

In [7]:
X = df.drop(columns=[target_col] + drop_cols)
y = df[target_col]

## 04. New features creation

In [8]:
X = X.copy()
X["Temperature difference [K]"] = (
    X["Process temperature [K]"] - X["Air temperature [K]"]
)

In [9]:
X["ToolWear_x_Torque"] = (
    X["Tool wear [min]"] * X["Torque [Nm]"]
)

here may be interesting to keep the relation value of the two variables that show a relationship with failures

## 05. Train/test split

In [13]:
#split test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y #use of stratification because the target is imbalanced
)

In [12]:
# check shapes and target balance
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (8000, 8)
X_test shape: (2000, 8)

Train target distribution:
Machine failure
0    0.966125
1    0.033875
Name: proportion, dtype: float64

Test target distribution:
Machine failure
0    0.966
1    0.034
Name: proportion, dtype: float64


# 06. Feature type identification

In [14]:
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Categorical features: ['Type']
Numeric features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature difference [K]', 'ToolWear_x_Torque']


# 07. Missing values check

In [15]:
print("Missing values in X_train:")
print(X_train.isna().sum().sort_values(ascending=False))

Missing values in X_train:
Type                          0
Air temperature [K]           0
Process temperature [K]       0
Rotational speed [rpm]        0
Torque [Nm]                   0
Tool wear [min]               0
Temperature difference [K]    0
ToolWear_x_Torque             0
dtype: int64


# 08. Define preprocessing pipeline

In [16]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [20]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [21]:
#integrate both pipelines in one
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# 09. Apply preprocessing

In [22]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

Scaling is included to support models sensitive to feature magnitude, such as logistic regression. Tree-based models do not require scaling, so this choice will be revisited if model-specific pipelines are created later.

In [23]:
X_train_prepared.head()


,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],num__Temperature difference [K],num__ToolWear_x_Torque,cat__Type_H,cat__Type_L,cat__Type_M
4058,0.998914,0.604282,-0.460607,0.718305,-0.843997,-1.100842,-0.621945,0.0,0.0,1.0
1221,-1.505194,-1.153260,-0.775574,0.638456,0.382263,1.299658,0.646140,0.0,0.0,1.0
6895,0.498092,1.077466,-1.007654,0.558607,0.460870,0.599512,0.689544,0.0,0.0,1.0
9863,-0.553633,-0.139294,-0.709265,1.626586,-0.372359,0.899575,0.151247,0.0,1.0,0.0
8711,-1.455112,-1.018064,1.070019,-1.128202,-0.906882,1.399679,-1.016909,0.0,1.0,0.0


In [25]:
print("Any missing values after preprocessing?")
print(X_train_prepared.isna().sum().sum())
print(X_test_prepared.isna().sum().sum())

Any missing values after preprocessing?
0
0


In [26]:
print("Prepared train shape:", X_train_prepared.shape)


Prepared train shape: (8000, 10)


In [27]:
print("Prepared test shape:", X_test_prepared.shape)

Prepared test shape: (2000, 10)


# 10. Exporting data sets

In [29]:

# Create output folder
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Export features
X_train_prepared.to_csv(output_dir / "X_train_prepared.csv", index=False)
X_test_prepared.to_csv(output_dir / "X_test_prepared.csv", index=False)

# Export targets
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.


# 11. Summary and next steps

Summary
- In this notebook, the dataset was prepared for binary classification of machine failure. Leakage-prone failure-mode columns and identifier columns were removed, a stratified train-test split was created, and a preprocessing pipeline was defined for numeric and categorical features. A small set of engineered features was added based on domain intuition and EDA findings: the temperature difference between process and air temperature, and a tool wear–torque interaction feature. Their usefulness will be validated during model development. The resulting transformed datasets are ready for model training and evaluation in the next notebook.

Next Steps
- Next, baseline classification models will be trained and compared, with particular attention to class imbalance and metrics beyond accuracy, such as precision, recall, F1-score, and ROC-AUC.